# Práctica 3 · UrbanIng-V2X

**Objetivo:** detectar vehículos, generar sus bounding boxes 3D y contar los presentes en la intersección.

Se utilizan únicamente los LiDAR de infraestructura **11, 12, 31 y 32**. **Sin tracking:** cada fotograma se procesa de forma independiente.

## Imports

In [ ]:
import base64
import csv
import html
import json
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from functools import cached_property
from pathlib import Path
from typing import ClassVar

import numpy as np
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.ndimage import gaussian_filter, median_filter
from scipy.spatial import cKDTree
from shapely import intersects_xy
from shapely.geometry import LineString, MultiPoint, Polygon, box as rectangle
from shapely.ops import unary_union

from IPython.display import HTML, display

# Estilo común de las figuras
plt.rcParams.update({'figure.dpi': 115, 'font.size': 10, 'axes.titlesize': 12})

## Parámetros

In [ ]:
# Rutas (el notebook se ejecuta desde la carpeta del proyecto)
ROOT = Path.cwd()
DATASET_DIR = ROOT / 'data' / 'dataset'
MAP_FILE = ROOT / 'data' / 'crossings_lanelet2map.osm'
CONFIG_FILE = ROOT / 'detector_config.json'
RESULTS_DIR = ROOT / 'results' / 'improved'
VIDEOS_DIR = ROOT / 'results' / 'videos'
assert CONFIG_FILE.is_file(), 'Para reejecutar, abrir el notebook desde la carpeta del proyecto.'

# Secuencias y LiDAR de infraestructura
SEQUENCES = ('20241126_0024_crossing1_09',
             '20241126_0008_crossing1_01',
             '20241127_0000_crossing1_00')
SENSORS = tuple(f'crossing1_{i}_lidar' for i in (11, 12, 31, 32))
GPS_ORIGIN = (11.438043, 48.771731)  # lon, lat del origen de crossing1 (devkit urbaning)

# Parámetros de preprocesamiento
ROI = (-45, 45, -45, 45)      # xmin, xmax, ymin, ymax en metros (coordenadas globales)
VOXEL_SIZE = 0.15             # tamaño del vóxel en metros
FOREGROUND_MAX_HEIGHT = 4.5   # altura máxima sobre el suelo de un punto de vehículo (m)
ROAD_MARGIN = 0.35            # margen de la máscara vial (m)
ROAD_CONTEXT_MARGIN = 0.5     # margen extra para no cortar carrocerías en el borde (m)

# Parámetros del detector: valores por defecto sobrescritos por detector_config.json
# (la misma combinación que usa la CLI)
DETECTOR_DEFAULTS = {'eps': 0.85, 'min_points': 4, 'min_height': 0.4,
                     'min_length': 1.0, 'min_width': 0.3, 'complete_boxes': True}
DETECTOR_CONFIG = {**DETECTOR_DEFAULTS,
                   **json.loads(CONFIG_FILE.read_text(encoding='utf-8'))['parameters']}

# Parámetros de presentación
DEMO_SEQUENCE = SEQUENCES[1]
DEMO_FRAME = 100
FRAMES_PER_SEQUENCE = 200
VIDEO_FPS = 10

## Carga y fusión de los LiDAR

In [ ]:
@dataclass
class LidarFrame:
    sequence: str
    index: int
    timestamp_ms: int
    sources: dict   # sensor -> archivo sincronizado
    clouds: dict    # sensor -> nube en coordenadas globales

    @cached_property
    def points(self):
        # Las cuatro nubes se fusionan antes de detectar para no duplicar vehículos.
        return np.concatenate(list(self.clouds.values()))


class SequenceDataset:
    """Fotogramas sincronizados de una secuencia, fusionados en coordenadas globales."""

    def __init__(self, name, root=DATASET_DIR):
        self.name = name
        self.folder = Path(root) / name
        self.calibration = json.loads((self.folder / 'calibration.json').read_text())
        with (self.folder / 'timesync_info.csv').open(newline='') as stream:
            self.sync = {row[0]: row[1:] for row in csv.reader(stream)}

    def __len__(self):
        return len(self.sync['timestamp_ms'])

    def frame(self, index):
        sources = {sensor: self.filename(sensor, index) for sensor in SENSORS}
        clouds = {sensor: self.sensor_cloud(sensor, index) for sensor in SENSORS}
        return LidarFrame(self.name, index, int(self.sync['timestamp_ms'][index]), sources, clouds)

    def sensor_cloud(self, sensor, index):
        """Nube de un LiDAR en coordenadas globales, sin puntos no finitos."""
        path = self.folder / sensor / self.filename(sensor, index)
        with np.load(path, allow_pickle=False) as data:
            xyz = np.column_stack([data[k] for k in ('x', 'y', 'z')])
        xyz = xyz[np.isfinite(xyz).all(axis=1)]
        return self.transform(xyz, self.calibration[sensor]['extrinsics']['gTl'])

    def filename(self, sensor, index):
        filename = self.sync[sensor][index]
        # Solo nombres simples: evita leer fuera de la carpeta del sensor.
        if not filename or Path(filename).name != filename:
            raise ValueError(f'Archivo inválido: {sensor}, frame {index}')
        return filename

    @staticmethod
    def transform(points, matrix):
        """p_global = R · p_LiDAR + t (convención de LidarData en el devkit)."""
        matrix = np.asarray(matrix, dtype=float)
        if matrix.shape != (4, 4) or not np.isfinite(matrix).all():
            raise ValueError('Calibración inválida')
        return points @ matrix[:3, :3].T + matrix[:3, 3]

## Mapa vial

In [ ]:
class RoadMap:
    """Calzada y ejes de carril del mapa lanelet2 oficial; no usa anotaciones de objetos."""
    AXIS_STEP = 2       # separación aproximada entre muestras del eje de carril (m)
    AXIS_RADIUS = 100   # solo se conservan ejes cercanos al cruce (m)

    def __init__(self, path=MAP_FILE, roi=ROI, margin=ROAD_MARGIN):
        # El OSM se lee una sola vez para la calzada y para los ejes de carril.
        self.lanelets = self._read_lanelets(path)
        self.extent = rectangle(roi[0], roi[2], roi[1], roi[3])
        self.polygon = self._road_polygon(margin)

    @cached_property
    def context(self):
        """Calzada ampliada para no cortar la carrocería al borde de la calzada."""
        return self.polygon.buffer(ROAD_CONTEXT_MARGIN)

    @cached_property
    def lane_segments(self):
        """Segmentos [x0, y0, x1, y1] de los ejes de carril."""
        segments = []
        for left, right in self.lanelets:
            axis = self._lane_axis(left, right)
            for start, end in zip(axis[:-1], axis[1:]):
                near = np.linalg.norm((start + end) / 2) < self.AXIS_RADIUS
                if np.linalg.norm(end - start) > 0.01 and near:
                    segments.append([*start, *end])
        return np.array(segments)

    def inside(self, points, area=None):
        points = np.asarray(points)
        return intersects_xy(self.polygon if area is None else area, points[:, 0], points[:, 1])

    def filter_boxes(self, boxes):
        """Conserva las cajas cuyo centro cae sobre la calzada."""
        if not boxes:
            return []
        mask = self.inside(np.array([b['center'][:2] for b in boxes]))
        return [b for b, valid in zip(boxes, mask) if valid]

    def lane_heading(self, xy):
        """Dirección del eje de carril más cercano al punto xy."""
        start, end = self.lane_segments[:, :2], self.lane_segments[:, 2:]
        direction = end - start
        t = np.clip(np.sum((xy - start) * direction, axis=1) / np.sum(direction ** 2, axis=1), 0, 1)
        nearest = np.argmin(np.linalg.norm(start + t[:, None] * direction - xy, axis=1))
        return np.arctan2(direction[nearest, 1], direction[nearest, 0])

    def draw(self, ax):
        polygons = list(self.polygon.geoms) if hasattr(self.polygon, 'geoms') else [self.polygon]
        for polygon in polygons:
            for ring in (polygon.exterior, *polygon.interiors):
                ax.plot(*ring.xy, color='tab:blue', linewidth=0.9)

    def _road_polygon(self, margin):
        polygons = [Polygon(np.concatenate((left, right[::-1]))).buffer(0)
                    for left, right in self.lanelets]
        polygons = [polygon for polygon in polygons if polygon.intersects(self.extent)]
        if not polygons:
            raise ValueError('No hay carriles viales dentro de la ROI')
        return unary_union(polygons).buffer(margin).intersection(self.extent)

    @classmethod
    def _lane_axis(cls, left, right):
        """Puntos medios entre ambos bordes del carril, cada ~2 m."""
        a, b = LineString(left), LineString(right)
        count = max(2, int(max(a.length, b.length) / cls.AXIS_STEP) + 1)
        return np.array([(np.array(a.interpolate(t, normalized=True).coords[0])
                          + np.array(b.interpolate(t, normalized=True).coords[0])) / 2
                         for t in np.linspace(0, 1, count)])

    @staticmethod
    def _read_lanelets(path):
        """Bordes (izquierdo, derecho) de los lanelets viales, en UTM32 relativo al origen (m)."""
        root = ET.parse(path).getroot()
        project = Transformer.from_crs('EPSG:4326', 'EPSG:32632', always_xy=True)
        origin = np.array(project.transform(*GPS_ORIGIN))
        nodes = {n.get('id'): np.array(project.transform(float(n.get('lon')), float(n.get('lat'))))
                 - origin for n in root.findall('node')}
        ways = {w.get('id'): np.array([nodes[n.get('ref')] for n in w.findall('nd')])
                for w in root.findall('way')}
        lanelets = []
        for relation in root.findall('relation'):
            tags = {t.get('k'): t.get('v') for t in relation.findall('tag')}
            if tags.get('type') != 'lanelet' or tags.get('subtype') != 'road':
                continue
            bounds = {m.get('role'): ways[m.get('ref')] for m in relation.findall('member')
                      if m.get('type') == 'way' and m.get('role') in ('left', 'right')}
            left, right = bounds['left'], bounds['right']
            # Ambos bordes deben recorrerse en el mismo sentido.
            if np.linalg.norm(left[0] - right[0]) > np.linalg.norm(left[0] - right[-1]):
                right = right[::-1]
            lanelets.append((left, right))
        return lanelets

## Suelo y filtrado de la nube

In [ ]:
@dataclass
class GroundModel:
    """Plano global z = ax + by + c (RANSAC) corregido con una rejilla local de 3 m."""
    CELL: ClassVar[float] = 3.0   # lado de la celda de corrección local (m)

    plane: np.ndarray
    grid: np.ndarray
    origin: np.ndarray

    @classmethod
    def fit(cls, points):
        plane = cls._ransac_plane(points)
        return cls(plane, *cls._local_grid(points, plane))

    def height(self, points):
        """Altura sobre el suelo local de los puntos usados en el ajuste."""
        cells = self._cells(points[:, :2], self.origin)
        return self._residual(points, self.plane) - self.grid[tuple(cells.T)]

    def bottom(self, xy):
        """Cota del suelo bajo un centro XY (se usa la celda más próxima de la rejilla)."""
        index = np.clip(self._cells(xy, self.origin), 0, np.array(self.grid.shape) - 1)
        return float(np.r_[xy, 1] @ self.plane + self.grid[tuple(index)])

    @classmethod
    def _cells(cls, xy, origin):
        return np.floor((xy - origin) / cls.CELL).astype(int)

    @staticmethod
    def _residual(points, plane):
        return points[:, 2] - points[:, :2] @ plane[:2] - plane[2]

    @staticmethod
    def _ransac_plane(points):
        """RANSAC sobre los mínimos de celdas XY de 2 m (semilla fija, reproducible)."""
        if len(points) < 30:
            raise ValueError('No hay suficientes puntos para estimar el suelo')
        _, groups = np.unique(np.floor(points[:, :2] / 2).astype(int), axis=0, return_inverse=True)
        order = np.lexsort((points[:, 2], groups))
        low = points[order[np.r_[True, np.diff(groups[order]) != 0]]]
        if len(low) < 10:
            raise ValueError('Cobertura insuficiente para estimar el suelo')
        design = np.column_stack((low[:, :2], np.ones(len(low))))
        rng = np.random.default_rng(7)
        best = np.zeros(len(low), dtype=bool)
        for _ in range(150):
            ids = rng.choice(len(low), 3, replace=False)
            if np.linalg.matrix_rank(design[ids]) != 3:
                continue
            model = np.linalg.solve(design[ids], low[ids, 2])
            if np.linalg.norm(model[:2]) > 0.15:   # pendiente máxima admitida
                continue
            inliers = np.abs(design @ model - low[:, 2]) < 0.18
            if inliers.sum() > best.sum():
                best = inliers
        if best.sum() < 10:
            raise ValueError('No se ha podido estimar el suelo')
        return np.linalg.lstsq(design[best], low[best, 2], rcond=None)[0]

    @classmethod
    def _local_grid(cls, points, plane):
        """Mínimo residuo por celda, relleno por vecino más cercano y suavizado."""
        origin = np.floor(points[:, :2].min(axis=0) / cls.CELL) * cls.CELL
        indices = cls._cells(points[:, :2], origin)
        residual = cls._residual(points, plane)
        grid = np.full(tuple(indices.max(axis=0) + 1), np.inf)
        # Solo cotas cercanas al plano inicial, para no tomar vehículos como suelo.
        valid = (residual > -0.4) & (residual < 0.3)
        np.minimum.at(grid, tuple(indices[valid].T), residual[valid])
        known = np.isfinite(grid)
        if not known.any():
            grid[:] = 0
            return grid, origin
        locations = np.column_stack(np.nonzero(known))
        missing = np.column_stack(np.nonzero(~known))
        if len(missing):
            nearest = cKDTree(locations).query(missing)[1]
            grid[tuple(missing.T)] = grid[tuple(locations[nearest].T)]
        return gaussian_filter(median_filter(grid, size=3), sigma=1), origin


@dataclass
class FilteredCloud:
    """Etapas intermedias del filtrado de un fotograma."""
    cropped: np.ndarray      # puntos dentro de la ROI
    reduced: np.ndarray      # un punto por vóxel
    foreground: np.ndarray   # puntos elevados próximos a la calzada
    ground: GroundModel


class PointCloudFilter:
    """ROI, vóxeles, suelo y máscara vial; lo comparten la demostración y el detector."""

    def __init__(self, road, min_height, roi=ROI, voxel=VOXEL_SIZE):
        self.road = road
        self.roi = roi
        self.voxel = voxel
        self.min_height = min_height

    def apply(self, points):
        cropped = self.crop(points, self.roi)
        reduced = self.voxelize(cropped, self.voxel)
        ground = GroundModel.fit(reduced)
        height = ground.height(reduced)
        # No se exige movimiento: los vehículos detenidos también se conservan.
        mask = ((height >= self.min_height) & (height <= FOREGROUND_MAX_HEIGHT)
                & self.road.inside(reduced, self.road.context))
        return FilteredCloud(cropped, reduced, reduced[mask], ground)

    @staticmethod
    def crop(points, roi):
        xmin, xmax, ymin, ymax = roi
        return points[(points[:, 0] >= xmin) & (points[:, 0] <= xmax)
                      & (points[:, 1] >= ymin) & (points[:, 1] <= ymax)]

    @staticmethod
    def voxelize(points, size):
        """Un punto por vóxel (el primero), conservando el orden original."""
        if not len(points):
            return points
        _, indices = np.unique(np.floor(points / size).astype(np.int64), axis=0, return_index=True)
        return points[np.sort(indices)]

## Agrupamiento y ajuste de cajas

In [ ]:
class XYClusterer:
    """DBSCAN en el plano XY con vecinos calculados bajo demanda."""

    def __init__(self, eps, min_points):
        self.eps = eps
        self.min_points = min_points

    def groups(self, points):
        if not len(points):
            return
        tree = cKDTree(points[:, :2])
        core = tree.query_ball_point(points[:, :2], self.eps, return_length=True) >= self.min_points
        labels = np.full(len(points), -1, dtype=int)
        cluster_id = 0
        for seed in np.flatnonzero(core):
            if labels[seed] >= 0:
                continue
            labels[seed] = cluster_id
            stack = [seed]
            while stack:
                current = stack.pop()
                neighbors = np.asarray(tree.query_ball_point(points[current, :2], self.eps))
                new = neighbors[labels[neighbors] < 0]
                labels[new] = cluster_id
                stack.extend(new[core[new]].tolist())
            yield points[labels == cluster_id]
            cluster_id += 1


class BoxFitter:
    """Caja orientada de un clúster, con base en el suelo local y completado de caras ocultas."""
    MIN_DIMENSIONS = (4.0, 1.8)   # largo y ancho mínimos al completar (m)
    SENSOR_MASTS = np.array([[-20.27, -2.59], [18.30, 1.46]])   # mástiles de los LiDAR (XY global)
    PLACEMENTS = ('away', 'centered', 'opposite')

    def __init__(self, ground, road=None, complete=True):
        self.ground = ground
        self.road = road   # RoadMap: su eje de carril orienta los clústeres pequeños
        self.complete = complete

    def fit(self, points, yaw=None, placement='away'):
        hull = MultiPoint(points[:, :2]).minimum_rotated_rectangle
        if hull.geom_type != 'Polygon':
            return None
        if yaw is None:
            yaw = self._heading(hull, points)
        rotation = self.rotation(yaw)
        local = points[:, :2] @ rotation
        lo, hi = local.min(axis=0), local.max(axis=0)
        observed = hi - lo
        center = (lo + hi) / 2
        observed_center = center @ rotation.T
        dimensions = observed.copy()
        if self.complete:
            # Priorización conservadora para carrocerías parcialmente visibles.
            dimensions = np.maximum(dimensions, self.MIN_DIMENSIONS)
            center = self._place(center, rotation, dimensions, observed, placement)
        xy = center @ rotation.T
        bottom = self.ground.bottom(xy)
        height = float(np.percentile(points[:, 2], 98) - bottom)
        return {'center': [float(xy[0]), float(xy[1]), bottom + height / 2],
                'dimensions': [float(dimensions[0]), float(dimensions[1]), height],
                'observed_dimensions_xy': observed.tolist(),
                'observed_center_xy': observed_center.tolist(),
                'yaw_rad': float((yaw + np.pi / 2) % np.pi - np.pi / 2),
                'class': 'vehicle_candidate', 'num_points': len(points)}

    def _heading(self, hull, points):
        """Lado largo del rectángulo mínimo; con poco soporte, dirección del carril."""
        corners = np.array(hull.exterior.coords)[:4]
        edges = np.roll(corners, -1, axis=0) - corners
        lengths = np.linalg.norm(edges, axis=1)
        edge = edges[np.argmax(lengths)]
        yaw = np.arctan2(edge[1], edge[0])
        if self.road is not None and (lengths.max() < 3.2 or lengths.min() < 0.8):
            yaw = self.road.lane_heading(points[:, :2].mean(axis=0))
        return yaw

    def _place(self, center, rotation, dimensions, observed, placement):
        """Desplaza la caja completada respecto al mástil más cercano (cara oculta)."""
        xy = center @ rotation.T
        nearest = np.argmin(np.linalg.norm(self.SENSOR_MASTS - xy, axis=1))
        viewpoint = self.SENSOR_MASTS[nearest] @ rotation
        shift = np.sign(center - viewpoint) * (dimensions - observed) / 2
        if placement == 'away':
            return center + shift
        if placement == 'opposite':
            return center - shift
        if placement != 'centered':
            raise ValueError('Colocación de caja desconocida')
        return center

    @staticmethod
    def rotation(yaw):
        return np.array([[np.cos(yaw), -np.sin(yaw)], [np.sin(yaw), np.cos(yaw)]])

    @classmethod
    def footprint(cls, box):
        """Huella BEV orientada de una caja métrica."""
        corners = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]])
        half = np.array(box['dimensions'][:2]) / 2
        return Polygon(corners * half @ cls.rotation(box['yaw_rad']).T + box['center'][:2])

## Fusión de fragmentos y detector

In [ ]:
@dataclass
class Fragment:
    """Grupo de puntos, su caja y cuántos clústeres DBSCAN lo forman."""
    points: np.ndarray
    box: dict
    count: int = 1


class FragmentResolver:
    """Fusión espacial de fragmentos y completado de cajas sin invadir las de más evidencia.

    Solo usa puntos del instante actual: no hay etiquetas ni asociaciones temporales.
    """

    def __init__(self, fitter, config):
        self.fitter = fitter
        self.config = config

    def resolve(self, groups):
        boxes = map(self.fitter.fit, groups)
        fragments = [Fragment(group, box) for group, box in zip(groups, boxes) if box is not None]
        if self.config.get('merge_fragments', True):
            fragments = self._merge(fragments)
        kept, polygons = [], []
        for fragment in sorted(fragments, key=lambda f: f.box['num_points'], reverse=True):
            original = fragment.box
            if not self.plausible(original):
                continue
            chosen = self._resolve_overlap(fragment.points, original, polygons)
            polygon = BoxFitter.footprint(chosen)
            # El solapamiento fuerte restante se considera una hipótesis duplicada.
            if self.is_duplicate(polygon, polygons, self.config.get('nms_overlap', 0.5)):
                continue
            chosen['fragments_merged'] = fragment.count
            chosen['overlap_resolved'] = chosen is not original
            kept.append(chosen)
            polygons.append(polygon)
        return kept

    def plausible(self, box):
        """Dimensiones observadas compatibles con un vehículo."""
        length, width = box['observed_dimensions_xy']
        return (self.config['min_length'] <= length <= 18
                and self.config['min_width'] <= width <= 3.5
                and 0.65 <= box['dimensions'][2] <= 4.5)

    @staticmethod
    def is_duplicate(polygon, polygons, overlap):
        return any(polygon.intersection(q).area / min(polygon.area, q.area) > overlap
                   for q in polygons)

    @staticmethod
    def _axis_angle(a, b):
        """Diferencia entre los ejes de dos cajas, en [0, π/2] (el sentido no importa)."""
        delta = 2 * (a['yaw_rad'] - b['yaw_rad'])
        return abs(np.arctan2(np.sin(delta), np.cos(delta))) / 2

    def _merge(self, fragments):
        """Une parejas compatibles hasta que no quede ninguna (siempre la primera encontrada)."""
        while (merge := self._find_merge(fragments)) is not None:
            i, j, fragment = merge
            fragments[i] = fragment
            del fragments[j]
        return fragments

    def _find_merge(self, fragments):
        for i in range(len(fragments)):
            for j in range(i + 1, len(fragments)):
                merged = self._try_merge(fragments[i], fragments[j])
                if merged is not None:
                    return i, j, merged
        return None

    def _try_merge(self, first, second):
        a, b = first.box, second.box
        if first.count + second.count > 3:
            return None
        if self._axis_angle(a, b) > np.deg2rad(25):
            return None
        pa, pb = BoxFitter.footprint(a), BoxFitter.footprint(b)
        if pa.intersection(pb).area / min(pa.area, pb.area) < 0.12:
            return None
        if cKDTree(first.points[:, :2]).query(second.points[:, :2])[0].min() > 1.8:
            return None
        combined = np.concatenate((first.points, second.points))
        merged = self.fitter.fit(combined)
        if merged is None:
            return None
        length, width = merged['observed_dimensions_xy']
        if length > 7.0 or width > 2.8:
            return None
        # La unión debe explicar un único vehículo, sin gran superficie vacía.
        if BoxFitter.footprint(merged).area > 1.3 * (pa.area + pb.area):
            return None
        return Fragment(combined, merged, first.count + second.count)

    def _resolve_overlap(self, group, original, polygons):
        """Elige la colocación de menor coste si la caja invade otras ya aceptadas."""
        polygon = BoxFitter.footprint(original)
        if not self.config.get('resolve_overlap', True) or not any(
                polygon.intersection(q).area > 0.15 for q in polygons):
            return original
        proposals = [(0.0, original), *self._proposals(group, original)]
        return min(proposals, key=lambda proposal: self._cost(proposal, polygons))[1]

    def _proposals(self, group, original):
        yaws = [original['yaw_rad']]
        # Con poco soporte, el carril más cercano puede tener la dirección perpendicular.
        if max(original['observed_dimensions_xy']) < 3.2:
            yaws.append(original['yaw_rad'] + np.pi / 2)
        for yaw in yaws:
            for placement in BoxFitter.PLACEMENTS:
                candidate = self.fitter.fit(group, yaw, placement)
                if candidate is None or candidate['dimensions'][1] > 3.5:
                    continue
                shift = np.linalg.norm(np.array(candidate['center'][:2]) - original['center'][:2])
                yield 0.05 * shift + (0.12 if yaw != original['yaw_rad'] else 0), candidate

    @staticmethod
    def _cost(proposal, polygons):
        prior, box = proposal
        polygon = BoxFitter.footprint(box)
        return prior + 10 * sum(polygon.intersection(q).area for q in polygons)


class VehicleDetector:
    """Detección geométrica por fotograma: DBSCAN XY, cajas orientadas y refinamiento espacial."""

    def __init__(self, road, config=DETECTOR_CONFIG):
        self.road = road
        self.config = config
        self.clusterer = XYClusterer(self.config['eps'], self.config['min_points'])

    def detect(self, cloud):
        groups = list(self.clusterer.groups(cloud.foreground))
        # El ajuste depende del suelo de cada fotograma.
        fitter = BoxFitter(cloud.ground, self.road, self.config['complete_boxes'])
        resolver = FragmentResolver(fitter, self.config)
        if self.config.get('spatial_refinement', True):
            return self.road.filter_boxes(resolver.resolve(groups))
        boxes = [box for box in map(fitter.fit, groups)
                 if box is not None and resolver.plausible(box)]
        return self._suppress_duplicates(self.road.filter_boxes(boxes))

    @staticmethod
    def _suppress_duplicates(boxes, overlap=0.5):
        """Supresión de cajas duplicadas por fragmentos; no usa identidades temporales."""
        kept, polygons = [], []
        for box in sorted(boxes, key=lambda b: b['num_points'], reverse=True):
            polygon = BoxFitter.footprint(box)
            if FragmentResolver.is_duplicate(polygon, polygons, overlap):
                continue
            kept.append(box)
            polygons.append(polygon)
        return kept

## Pipeline de procesamiento

In [ ]:
@dataclass
class FrameAnalysis:
    """Resultado de un fotograma: nube fusionada, etapas de filtrado y cajas."""
    frame: LidarFrame
    cloud: FilteredCloud
    boxes: list

    @property
    def count(self):
        return len(self.boxes)


class DetectionPipeline:
    """Fusión -> filtrado -> detección de cada fotograma, de forma independiente (sin tracking)."""

    def __init__(self, sequence=DEMO_SEQUENCE, config=DETECTOR_CONFIG):
        # Una configuración parcial se completa con los valores por defecto del detector.
        config = {**DETECTOR_DEFAULTS, **config}
        self.dataset = SequenceDataset(sequence)
        self.road = RoadMap(MAP_FILE)
        self.filter = PointCloudFilter(self.road, config['min_height'])
        self.detector = VehicleDetector(self.road, config)

    def run(self, indices=None):
        for index in (range(len(self.dataset)) if indices is None else indices):
            yield self.analyze(index)

    def analyze(self, index):
        return self._process(self.dataset.frame(index))

    def _process(self, frame):
        cloud = self.filter.apply(frame.points)
        return FrameAnalysis(frame, cloud, self.detector.detect(cloud))

## Resultados exportados

In [ ]:
@dataclass
class SequenceResults:
    """Detecciones por fotograma exportadas por la CLI (main.py) para una secuencia."""
    name: str
    records: list

    @classmethod
    def load(cls, name, root=RESULTS_DIR):
        path = Path(root) / name / 'detections.jsonl'
        if not path.is_file():
            raise FileNotFoundError(f'Falta {path}: ejecutar antes "uv run python main.py"')
        records = [json.loads(line) for line in path.read_text().splitlines()]
        if len(records) != FRAMES_PER_SEQUENCE:
            raise ValueError(f'Se esperan {FRAMES_PER_SEQUENCE} fotogramas en {name}')
        if any(r['count'] != len(r['boxes']) for r in records):
            raise ValueError(f'El conteo no coincide con las cajas en {name}')
        return cls(name, records)

    def __len__(self):
        return len(self.records)

    @property
    def counts(self):
        return [r['count'] for r in self.records]

    @property
    def times_s(self):
        start = self.records[0]['timestamp_ms']
        return [(r['timestamp_ms'] - start) / 1000 for r in self.records]

    def count_at(self, frame_index):
        return next(r['count'] for r in self.records if r['frame_index'] == frame_index)

## Visualización

In [ ]:
class SceneVisualizer:
    """Tablas y figuras cenitales de la presentación."""
    MAX_POINTS = 50000      # submuestreo aproximado de las nubes dibujadas
    SENSOR_VOXEL = 0.45     # vóxel solo para dibujar cada LiDAR por separado
    TH_STYLE = 'text-align:left;padding:8px'
    TD_STYLE = 'padding:8px;border-bottom:1px solid #ddd'

    def __init__(self, road, roi=ROI):
        self.road = road
        self.roi = roi

    @classmethod
    def table(cls, headers, rows):
        """Tabla HTML sencilla con los valores escapados."""
        header = cls._cells('th', cls.TH_STYLE, headers)
        body = ''.join('<tr>' + cls._cells('td', cls.TD_STYLE, row) + '</tr>' for row in rows)
        display(HTML('<table style="border-collapse:collapse"><thead><tr>' + header
                     + '</tr></thead><tbody>' + body + '</tbody></table>'))

    @staticmethod
    def _cells(tag, style, values):
        return ''.join(f'<{tag} style="{style}">{html.escape(str(v))}</{tag}>' for v in values)

    def show_fusion(self, frame):
        self.table(['Sensor', 'Archivo sincronizado'], frame.sources.items())
        print(f'Fotograma {frame.index} · timestamp {frame.timestamp_ms} ms · '
              f'{len(frame.points):,} puntos fusionados')
        fig, ax = plt.subplots(figsize=(10, 8))
        for sensor, cloud in frame.clouds.items():
            xyz = PointCloudFilter.crop(cloud, self.roi)
            xyz = PointCloudFilter.voxelize(xyz, self.SENSOR_VOXEL)
            ax.scatter(xyz[:, 0], xyz[:, 1], s=0.7, alpha=0.6, label=sensor)
        self._bev_axes(ax, 'Cuatro puntos de vista en un sistema común',
                       'X global (m)', 'Y global (m)')
        ax.legend(markerscale=5, fontsize=8, loc='upper left')
        self._show(fig)

    def show_filtering(self, cloud):
        self.table(['Etapa', 'Puntos'], [('Dentro de la ROI', len(cloud.cropped)),
                                         ('Tras vóxeles', len(cloud.reduced)),
                                         ('Elevados cerca de calzada', len(cloud.foreground))])
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        titles = ['Nube reducida y máscara vial', 'Puntos para el agrupamiento']
        for ax, points, title in zip(axes, [cloud.reduced, cloud.foreground], titles):
            sample = self._sample(points)
            ax.scatter(sample[:, 0], sample[:, 1], s=0.6, color='#536878')
            self.road.draw(ax)
            self._bev_axes(ax, title)
        self._show(fig)

    def show_detection(self, analysis):
        boxes = analysis.boxes
        print(f'Conteo estimado en este fotograma: {len(boxes)} candidatos a vehículo')
        self.table(['Caja local', 'Centro XYZ (m)', 'Largo × ancho × alto (m)', 'Giro (rad)'],
                   [self._box_row(i, b) for i, b in enumerate(boxes[:6], 1)])
        print('Se muestran las primeras seis cajas; los índices no son identidades de tracking.')
        fig, ax = plt.subplots(figsize=(10, 8))
        sample = self._sample(analysis.cloud.reduced)
        ax.scatter(sample[:, 0], sample[:, 1], s=0.5, color='0.7')
        self.road.draw(ax)
        for i, b in enumerate(boxes, 1):
            ax.plot(*BoxFitter.footprint(b).exterior.xy, color='crimson', linewidth=1.1)
            ax.text(*b['center'][:2], str(i), fontsize=7)
        title = f'Detección final: {len(boxes)} cajas · fotograma {analysis.frame.index}'
        self._bev_axes(ax, title)
        self._show(fig)

    def show_counts(self, results, frame=DEMO_FRAME):
        headers = ['Secuencia', 'Fotogramas procesados',
                   f'Vehículos detectados en el fotograma {frame}']
        self.table(headers, [(r.name, len(r), r.count_at(frame)) for r in results])
        fig, axes = plt.subplots(len(results), 1, figsize=(12, 8),
                                 layout='constrained', squeeze=False)
        for ax, r in zip(axes[:, 0], results):
            ax.plot(r.times_s, r.counts, color='#146b9b', linewidth=1.5)
            ax.set(title=r.name, xlabel='Tiempo (s)', ylabel='Vehículos detectados', ylim=(0, None))
            ax.grid(alpha=0.2)
        self._show(fig, tight=False)

    def _sample(self, points):
        return points[::max(1, len(points) // self.MAX_POINTS)]

    def _bev_axes(self, ax, title, xlabel='X (m)', ylabel='Y (m)'):
        ax.set(xlim=self.roi[:2], ylim=self.roi[2:], aspect='equal',
               xlabel=xlabel, ylabel=ylabel, title=title)

    @staticmethod
    def _box_row(index, box):
        return (index, ', '.join(f'{v:.2f}' for v in box['center']),
                ' × '.join(f'{v:.2f}' for v in box['dimensions']), f"{box['yaw_rad']:.2f}")

    @staticmethod
    def _show(fig, tight=True):
        if tight:
            fig.tight_layout()
        display(fig)
        plt.close(fig)

In [ ]:
class VideoGallery:
    """Reproductores HTML de los vídeos de render_video.py, con avance fotograma a fotograma."""

    def __init__(self, folder=VIDEOS_DIR, sequences=SEQUENCES, fps=VIDEO_FPS,
                 frames=FRAMES_PER_SEQUENCE):
        self.folder = Path(folder)
        self.sequences = sequences
        self.fps = fps
        self.frames = frames

    def show(self):
        missing = [name for name in self.sequences if not (self.folder / f'{name}.mp4').is_file()]
        if missing:
            raise FileNotFoundError(f'Faltan vídeos en {self.folder}: {", ".join(missing)}. '
                                    'Ejecutar antes "uv run --extra video python render_video.py"')
        display(HTML(self.html()))

    def html(self):
        """Los vídeos se incrustan en base64 para que el notebook sea autocontenido."""
        cards = ''.join(self._card(index, name) for index, name in enumerate(self.sequences))
        return self._intro() + cards + '</div>' + self._script()

    def _source(self, name):
        video = (self.folder / f'{name}.mp4').read_bytes()
        return 'data:video/mp4;base64,' + base64.b64encode(video).decode('ascii')

    def _intro(self):
        return ('<div style="font-family:Arial,sans-serif">\n'
                f'<p>{self.frames} fotogramas por secuencia · {self.fps} fps · {self.frames // self.fps} segundos · sin tracking.</p>\n'
                '<p>Izquierda: nube LiDAR con el fondo original. Derecha: clústeres detectados, cajas y conteo estimado del instante.\n'
                'Puedes pausar, cambiar la velocidad y ampliar a pantalla completa.</p>\n'
                '<p>Todos los clústeres aceptados y sus cajas se muestran en naranja. Se conserva el fondo original: suelo gris y otros puntos elevados azul claro.\n'
                'El color indica detección; no identifica vehículos ni realiza tracking.</p>\n')

    def _card(self, index, name):
        vid = f'lidar-video-{index}'
        return f"""<section style="margin:24px 0">
<h3>{name}</h3>
<video id="{vid}" controls preload="metadata" playsinline style="width:100%;max-width:1400px" src="{self._source(name)}"></video>
<div style="display:flex;gap:12px;align-items:center;margin:8px 0">
<button onclick="stepFrame('{vid}',-1)">← Fotograma</button>
<button onclick="stepFrame('{vid}',1)">Fotograma →</button>
<label>Velocidad <select onchange="document.getElementById('{vid}').playbackRate=Number(this.value)">
<option value="0.25">0,25×</option><option value="0.5">0,5×</option><option value="1" selected>1×</option></select></label>
</div></section>"""

    def _script(self):
        return f"""<script>
function stepFrame(id, direction) {{
 const v=document.getElementById(id); v.pause();
 const frame=Math.floor(v.currentTime*{self.fps}+0.001);
 v.currentTime=(Math.max(0,Math.min({self.frames - 1},frame+direction))+0.01)/{self.fps};
}}
</script>"""

## 1. Datos

Secuencias del cruce `crossing1`:

- `20241126_0024_crossing1_09`
- `20241126_0008_crossing1_01`
- `20241127_0000_crossing1_00`

Los sensores 11 y 12 comparten ubicación, pero cada LiDAR tiene su propia calibración. No se utilizan LiDAR de vehículos ni cámaras.

In [ ]:
print('Sensores utilizados:', ', '.join(SENSORS))
print('Asociación entre fotogramas: desactivada (sin tracking).')

## 2. Sincronización y fusión de los cuatro LiDAR

Se seleccionan los archivos del mismo instante mediante `timesync_info.csv` y se transforma cada nube a coordenadas globales con su calibración: **`p_global = R · p_LiDAR + t`**.

Las cuatro nubes se fusionan antes de detectar, para evitar sumar detecciones independientes del mismo vehículo. Se muestra el fotograma 100 de `20241126_0008_crossing1_01`.

In [ ]:
pipeline = DetectionPipeline(DEMO_SEQUENCE)
analysis = pipeline.analyze(DEMO_FRAME)
visualizer = SceneVisualizer(pipeline.road)
visualizer.show_fusion(analysis.frame)

## 3. Filtrado de la nube

Se reduce la densidad de puntos con vóxeles, se estima la altura del suelo y se conservan los puntos elevados próximos a la calzada. El mapa vial delimita la zona de detección.

Los vehículos detenidos también se consideran: no se exige movimiento.

In [ ]:
visualizer.show_filtering(analysis.cloud)

## 4. Detección y bounding boxes

Se agrupan los puntos mediante DBSCAN y se unen fragmentos compatibles de un mismo vehículo. La unión debe cumplir restricciones de orientación, distancia y dimensiones.

Se ajustan cajas orientadas con base en el suelo local. Para completar observaciones parciales se comparan varias posiciones y orientaciones, conservando los puntos observados y penalizando invadir otras cajas. Se suprimen duplicados restantes.

**El conteo es el número de cajas aceptadas en el fotograma.** No se asocian vehículos entre instantes.

In [ ]:
clustering = {k: pipeline.detector.config[k] for k in ('eps', 'min_points')}
print('Parámetros de agrupamiento:', clustering)
visualizer.show_detection(analysis)

### Vídeos de las tres secuencias

Cada vídeo muestra 200 fotogramas a 10 fps. Se conserva el fondo original: suelo gris y puntos elevados azul claro. A la derecha, los puntos de todos los clústeres aceptados y sus cajas aparecen en el mismo naranja. El conteo corresponde a las cajas de cada instante, **sin tracking**. Se pueden pausar y reproducir a menor velocidad.

In [ ]:
VideoGallery().show()

## 5. Conteo en las tres secuencias

Conteos por fotograma obtenidos con los cuatro LiDAR de infraestructura. Cada punto de las gráficas es una detección independiente, **sin tracking**.

Son conteos estimados: pueden persistir falsas detecciones y omisiones. No se suman para obtener vehículos únicos.

In [ ]:
results = [SequenceResults.load(name) for name in SEQUENCES]
visualizer.show_counts(results)